# Baseline Representations

This notebook uses the shared `rarecell` modules to load data, preprocess RNA/protein modalities, compute baseline representations, and save benchmark-ready outputs (matching the script workflow).

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for _p in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from rarecell.config import DATA_DIR, FIGURES_DIR, REPRESENTATION_KEY_MAP, TABLES_DIR
from rarecell.io import get_cell_labels, get_protein_matrix, get_rna_adata, load_citeseq
from rarecell.plotting import plot_bar_counts, plot_embedding
from rarecell.preprocessing import align_cells_between_modalities, preprocess_protein, preprocess_rna
from rarecell.io import make_candidate_target_population_table
from rarecell.representations import (
    compute_joint_pca_representation,
    compute_protein_pca,
    compute_rna_pca,
    compute_umap_from_embedding,
    save_embedding,
)
from rarecell.utils import write_json

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)


Set `input_spec` to a raw or saved CITE-seq input (local file or scvi 10x spec).

In [ ]:
input_spec = "scvi:5k_pbmc_protein_v3_nextgem"
output_path = DATA_DIR / "processed" / "pbmc5k_10x_citeseq_representations.h5ad"
label_key = None
rna_pcs = 30
protein_pcs = 10
n_top_genes = 2000

In [ ]:
obj = load_citeseq(input_spec)
labels = get_cell_labels(obj, preferred_keys=[label_key] if label_key else None)
rna = get_rna_adata(obj)
protein = get_protein_matrix(obj)
rna, protein = align_cells_between_modalities(rna, protein)

if "counts" not in rna.layers:
    rna.layers["counts"] = rna.X.copy()
raw_protein = protein.copy()
if labels is not None:
    labels = labels.reindex(rna.obs_names)

print(f"Aligned RNA cells: {rna.n_obs:,}")
print(f"RNA genes: {rna.n_vars:,}")
print(f"Protein features: {protein.shape[1]:,}")

In [ ]:
rna_processed = preprocess_rna(rna, n_top_genes=n_top_genes, n_pcs=rna_pcs)
protein_processed = preprocess_protein(protein)
protein_processed = protein_processed.loc[rna_processed.obs_names].copy()
raw_protein = raw_protein.loc[rna_processed.obs_names].copy()

rna_pca = compute_rna_pca(rna_processed, n_components=rna_pcs)
protein_pca = compute_protein_pca(protein_processed, n_components=protein_pcs)
joint = compute_joint_pca_representation(rna_pca, protein_pca, scale_blocks=True)

rna_umap = compute_umap_from_embedding(rna_pca)
protein_umap = compute_umap_from_embedding(protein_pca)
joint_umap = compute_umap_from_embedding(joint)

In [ ]:
try:
    import scanpy as sc

    if "neighbors" not in rna_processed.uns and "X_pca" in rna_processed.obsm:
        n_pcs = min(rna_pcs, rna_processed.obsm["X_pca"].shape[1])
        sc.pp.neighbors(rna_processed, n_neighbors=min(15, rna_processed.n_obs - 1), n_pcs=n_pcs)
    if "leiden" not in rna_processed.obs and "neighbors" in rna_processed.uns:
        sc.tl.leiden(rna_processed, random_state=0)
except Exception as exc:
    warnings.warn(f"Leiden clustering failed; continuing without clusters. {exc}")

if labels is not None and not labels.dropna().empty:
    candidate_labels = labels.dropna().astype(str)
elif "leiden" in rna_processed.obs:
    candidate_labels = "cluster_" + rna_processed.obs["leiden"].astype(str)
    candidate_labels.name = "cell_type_candidate"
else:
    candidate_labels = None

if "leiden" in rna_processed.obs:
    cluster_counts = rna_processed.obs["leiden"].astype(str).value_counts().sort_index().rename_axis(
        "cluster").reset_index(name="n_cells")
    cluster_counts["fraction"] = cluster_counts["n_cells"] / cluster_counts["n_cells"].sum()
else:
    cluster_counts = pd.DataFrame(columns=["cluster", "n_cells", "fraction"])

candidate_table = make_candidate_target_population_table(candidate_labels)

In [ ]:
marker_table = pd.DataFrame()
if "leiden" in rna_processed.obs:
    try:
        import scanpy as sc

        sc.tl.rank_genes_groups(rna_processed, groupby="leiden", method="wilcoxon")
        result = rna_processed.uns.get("rank_genes_groups")
        if result is not None and "names" in result:
            groups = result["names"].dtype.names
            if groups is not None:
                rows = []
                optional = {
                    "scores": "score",
                    "pvals": "pval",
                    "pvals_adj": "pval_adj",
                    "logfoldchanges": "logfoldchange",
                }
                for group in groups:
                    for rank, gene in enumerate(result["names"][group][:20], start=1):
                        row = {"cluster": group, "rank": rank, "gene": gene}
                        for source_key, target_key in optional.items():
                            row[target_key] = result[source_key][group][rank - 1] if source_key in result else pd.NA
                        rows.append(row)
                marker_table = pd.DataFrame(rows)
    except Exception as exc:
        warnings.warn(f"Marker gene export failed; continuing without marker table. {exc}")


In [ ]:
save_embedding(rna_pca, TABLES_DIR / "rna_pca.csv")
save_embedding(protein_pca, TABLES_DIR / "protein_pca.csv")
save_embedding(joint, TABLES_DIR / "joint_pca.csv")
save_embedding(rna_umap, TABLES_DIR / "rna_umap.csv")
save_embedding(protein_umap, TABLES_DIR / "protein_umap.csv")
save_embedding(joint_umap, TABLES_DIR / "joint_umap.csv")
cluster_counts.to_csv(TABLES_DIR / "cluster_counts.csv", index=False)
candidate_table.to_csv(TABLES_DIR / "candidate_target_populations.csv", index=False)
if not marker_table.empty:
    marker_table.to_csv(TABLES_DIR / "rna_cluster_markers.csv", index=False)

run_parameters = {
    "input": str(input_spec),
    "output": str(output_path),
    "tables_dir": str(TABLES_DIR),
    "figures_dir": str(FIGURES_DIR),
    "rna_pcs": rna_pcs,
    "protein_pcs": protein_pcs,
    "n_top_genes": n_top_genes,
    "label_key": label_key,
    "n_aligned_cells": int(rna_processed.n_obs),
    "n_rna_features": int(rna_processed.n_vars),
    "n_protein_features": int(protein_processed.shape[1]),
    "leiden_succeeded": bool("leiden" in rna_processed.obs),
    "marker_export_succeeded": bool(not marker_table.empty),
}
write_json(run_parameters, TABLES_DIR / "run_parameters.json")

In [ ]:
if labels is not None:
    rna_processed.obs["cell_type_simple"] = labels.reindex(rna_processed.obs_names)
rna_processed.obsm["X_rna_pca"] = rna_pca.reindex(rna_processed.obs_names).to_numpy()
rna_processed.obsm["X_protein_pca"] = protein_pca.reindex(rna_processed.obs_names).to_numpy()
rna_processed.obsm[REPRESENTATION_KEY_MAP["joint_pca"]] = joint.reindex(rna_processed.obs_names).to_numpy()
rna_processed.obsm["X_rna_umap"] = rna_umap.reindex(rna_processed.obs_names).to_numpy()
rna_processed.obsm["X_protein_umap"] = protein_umap.reindex(rna_processed.obs_names).to_numpy()
rna_processed.obsm["X_joint_umap"] = joint_umap.reindex(rna_processed.obs_names).to_numpy()
rna_processed.obsm["protein_counts"] = raw_protein.reindex(rna_processed.obs_names).to_numpy()
rna_processed.uns["protein_names"] = list(raw_protein.columns.astype(str))
rna_processed.write_h5ad(output_path)

color_labels = labels if labels is not None else candidate_labels
plot_embedding(rna_umap, color_labels, "RNA-only PCA representation", FIGURES_DIR / "rna_umap.png")
plot_embedding(protein_umap, color_labels, "Protein-only PCA representation", FIGURES_DIR / "protein_umap.png")
plot_embedding(joint_umap, color_labels, "Joint RNA–protein representation", FIGURES_DIR / "joint_umap.png")
if labels is not None:
    plot_bar_counts(labels, FIGURES_DIR / "cell_label_counts.png", "Cell-label counts", "Cell label", "Number of cells")
if "leiden" in rna_processed.obs:
    plot_bar_counts(rna_processed.obs["leiden"].astype(str), FIGURES_DIR / "cluster_counts.png",
                    "Leiden cluster counts", "Cluster", "Number of cells")

In [ ]:
display(cluster_counts.head())
display(candidate_table.head())
if not marker_table.empty:
    display(marker_table.head())

for filename in ["rna_umap.png", "protein_umap.png", "joint_umap.png"]:
    path = FIGURES_DIR / filename
    if path.exists():
        print(filename)
        display(Image(filename=str(path)))



## Summary

In [ ]:
# Summary of generated outputs and deviations from make_baseline_representations.py
generated = [
    (output_path, "h5ad with all representations"),
    (TABLES_DIR / "rna_pca.csv", ""),
    (TABLES_DIR / "protein_pca.csv", ""),
    (TABLES_DIR / "joint_pca.csv", ""),
    (TABLES_DIR / "rna_umap.csv", ""),
    (TABLES_DIR / "protein_umap.csv", ""),
    (TABLES_DIR / "joint_umap.csv", ""),
    (TABLES_DIR / "cluster_counts.csv", "if Leiden succeeded"),
    (TABLES_DIR / "candidate_target_populations.csv", ""),
    (TABLES_DIR / "rna_cluster_markers.csv", "if markers succeeded"),
    (TABLES_DIR / "run_parameters.json", ""),
    (FIGURES_DIR / "rna_umap.png", ""),
    (FIGURES_DIR / "protein_umap.png", ""),
    (FIGURES_DIR / "joint_umap.png", ""),
]
print("Generated outputs:")
for p, note in generated:
    status = "OK" if Path(p).exists() else "MISSING"
    suffix = f"  # {note}" if note else ""
    try:
        rel = Path(p).relative_to(PROJECT_ROOT)
    except ValueError:
        rel = p
    print(f"  [{status}] {rel}{suffix}")
print()
print("Deviations from make_baseline_representations.py:")
print("  - No .h5mu output support (always writes .h5ad).")
print("  - No file-level logging (script uses setup_file_logger).")
